Project Drishti · Any data. Any domain. One grammar.

Copyright (c) 2026 Ashutosh Sinha <ajsinha@gmail.com>.
All rights reserved.

PROPRIETARY AND CONFIDENTIAL.

This file is the confidential and proprietary property of Ashutosh Sinha.
Unauthorised copying, use, modification, distribution or disclosure of this
file, via any medium, is strictly prohibited except with the express prior
written permission of the copyright holder.

See the LICENSE file in the root of this repository for the full terms.

# 04 · A pivot view, with Rachana paths over trade legs

The pivot engine (PIVOT_GUIDE.md) answers a view: rows, columns, values and filters, each a field or a Rachana path into the document. `$.LEGS[*].NOTIONAL` is the notional of every leg of a trade: a view that uses it has one row per leg, and a trade field in it is counted once per trade, never once per leg. Flight answers one row per cell and measure.

*Read only:* this notebook only reads. Flight refuses every call that would write (`DoPut`, `DoExchange`), and nothing here asks for one.

**Setup.** `flight_setup.py` (beside this notebook) finds the one-file client `clients/python/drishti_flight.py` and reads the address, the token and the CA from the environment: `DRISHTI_FLIGHT_URL`, `DRISHTI_TOKEN` (or `DRISHTI_TOKEN_FILE`) and `DRISHTI_FLIGHT_CA`. No token is ever written in a notebook.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))     # run Jupyter from examples/clients/python, where flight_setup.py is
from flight_setup import connect, meta, FlightRefused, SCOPE, DATE, STREAM_WAIT

## Legs' notional by desk

`$.TT_GROUP` is the trade's desk. The view is the same JSON as `POST /api/v1/pivot/query`.

In [ ]:
f = connect()
view = {"pivot": 1, "source": "trade",
        "rows": ["$.TT_GROUP"],
        "values": [{"measure": "sum($.LEGS[*].NOTIONAL)", "as": "Notional"}]}
if DATE:
    view["date"] = DATE
cells = f.pivot(view)
print("provenance:", meta(cells, "drishti.provenance"))
df = cells.to_pandas()
df.sort_values("VALUE", ascending=False).head(15)

## The answer's shape

Each row is a cell: `ROW` and `COLUMN` (the keys of the grid's row and column), one column per dimension, `MEASURE`, `VALUE` and `MEMBERS` (how many rows made it: here legs). Totals are rows too, made by the engine from their own members, so a client never adds cells to get a total.

In [ ]:
FIXED = {"ROW", "COLUMN", "MEASURE", "VALUE", "MEMBERS"}
dims = [c for c in df.columns if c not in FIXED]
print("dimensions:", dims)
df[dims + ["MEASURE", "VALUE", "MEMBERS"]].head()

## Desk by trade year: a grid

Add a column field (`$.TRADE_DATE:year`, the trade date by year) and lay the cells out as a grid in pandas.

In [ ]:
grid_view = dict(view, columns=["$.TRADE_DATE:year"])
g = f.pivot(grid_view).to_pandas()
gd = [c for c in g.columns if c not in FIXED]
grid = g.pivot_table(index=gd[0], columns=gd[1], values="VALUE", aggfunc="first") if len(gd) == 2 else g
(grid / 1e9).round(2)      # billions

## A filter on a list

A filter on a leg field keeps the trades with **any** leg matching: here the trades with a JPY leg, counted by product.

In [ ]:
jpy = {"pivot": 1, "source": "trade",
       "rows": ["$.PRODUCT_TYPE"],
       "values": [{"measure": "count()", "as": "Trades"}],
       "filters": {"$.LEGS[*].CURRENCY": {"in": ["JPY"]}}}
if DATE:
    jpy["date"] = DATE
f.pivot(jpy).to_pandas().sort_values("VALUE", ascending=False).head(10)

In [ ]:
f.close()